# SaaS Retention & Revenue Risk Analytics (Databricks)

Adaptado do kernel Kaggle de Bogdana Matrofailo para o workspace Databricks do challenge 001.

1. Rode o **LAB 01** (`01_LAB_importando_dados/lab01_carga_csv`) para criar `workspace.ravenstack.ravenstack_*`.
2. Widgets: `catalog_name`, `schema_name`, `data_path` (fallback CSV em `dados/` se a tabela Delta ainda nao existir).
3. Colunas = header dos CSVs. Label canonico de churn = `ravenstack_churn_events` (o notebook original tambem usa `accounts.churn_flag` em KPIs — trate como atributo, nao como verdade).
4. Cluster ou serverless → **Run All**.


# SaaS Retention & Revenue Risk Analytics

**A business analytics project focused on churn drivers, customer lifecycle behavior, product engagement, and revenue risk in a subscription-based SaaS company.**

## Business Problem

Customer churn is one of the most important challenges for SaaS companies because it directly affects recurring revenue, customer lifetime value, and future growth.

For RavenStack, a fictional AI-powered SaaS platform, the key business question is not only **how many customers churn**, but **which customers are most at risk, why they leave, and where the company should focus retention efforts first**.

This analysis focuses on five business questions:

- Which customer segments have the highest churn risk?
- How is product usage connected to retention?
- Do support experience and satisfaction influence churn?
- Which markets or industries create the highest revenue exposure?
- What actions can help the business reduce churn and protect recurring revenue?

The goal is to turn raw SaaS operational data into clear, actionable business insights.

## Dataset Overview

The dataset represents a fictional AI-powered SaaS platform called RavenStack and simulates a real-world subscription-based business environment.

It includes five connected operational tables:

- customer accounts
- subscription plans and revenue
- product feature usage
- support activity
- customer churn events

Together, these tables allow us to analyze the full customer lifecycle: from signup and product usage to support experience, subscription behavior, churn, and revenue exposure.

#### Dataset Credit

This is a fully synthetic dataset created by River @ Rivalytics and used for educational and portfolio purposes.

## Setup and Data Loading

Before starting the analysis, we first import the required Python libraries, apply a consistent visual style, and load all RavenStack datasets.

A consistent setup helps keep the notebook clean, reproducible, and visually aligned across all charts.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.io as pio
import plotly.express as px

In [ ]:
# Global visual style

sns.set_theme(
    style="darkgrid",
    context="notebook",
    palette="viridis"
)

plt.style.use("dark_background")

plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.facecolor": "#111111",
    "axes.facecolor": "#111111",
    "axes.edgecolor": "#333333",
    "axes.labelcolor": "white",
    "axes.titleweight": "bold",
    "axes.titlesize": 18,
    "axes.labelsize": 12,
    "xtick.color": "white",
    "ytick.color": "white",
    "text.color": "white",
    "grid.color": "#333333"
})

pio.templates.default = "plotly_dark"

### Load Source Tables

The project uses five related CSV files that represent different parts of the SaaS business: accounts, subscriptions, feature usage, support tickets, and churn events.

In [ ]:
from pathlib import Path
import pandas as pd

BOOL_COLUMNS = {
    "is_trial", "churn_flag", "upgrade_flag", "downgrade_flag",
    "auto_renew_flag", "is_beta_feature", "escalation_flag",
    "preceding_upgrade_flag", "preceding_downgrade_flag", "is_reactivation",
}
DATE_COLUMNS = {
    "signup_date", "start_date", "end_date", "usage_date", "churn_date",
    "submitted_at", "closed_at",
}


def widget_or_default(name, default):
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


def normalize_frame(df):
    for col in df.columns:
        if col in BOOL_COLUMNS:
            df[col] = df[col].map(
                lambda x: None if pd.isna(x) else str(x).strip().lower() in ("true", "1", "t")
            )
        elif col in DATE_COLUMNS:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def resolve_data_path(explicit):
    marker = "ravenstack_accounts.csv"
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    try:
        notebook_path = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        parts = [part for part in notebook_path.split("/") if part]
        if "notebook-data-001-churn" in parts:
            idx = parts.index("notebook-data-001-churn")
            repo = "/".join(parts[: idx + 1])
            candidates.append(Path("/Workspace") / repo / "dados")
    except Exception:
        pass
    here = Path.cwd()
    candidates.extend([here / "dados", here.parent / "dados", here.parent.parent / "dados"])
    for candidate in candidates:
        if (candidate / marker).exists():
            return str(candidate)
    raise FileNotFoundError("Rode o LAB 01 ou defina data_path para a pasta dados/.")


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "ravenstack")
data_path_widget = widget_or_default("data_path", "")

try:
    accounts = spark.table(f"{catalog_name}.{schema_name}.ravenstack_accounts").toPandas()
    subscriptions = spark.table(f"{catalog_name}.{schema_name}.ravenstack_subscriptions").toPandas()
    feature_usage = spark.table(f"{catalog_name}.{schema_name}.ravenstack_feature_usage").toPandas()
    support_tickets = spark.table(f"{catalog_name}.{schema_name}.ravenstack_support_tickets").toPandas()
    churn_events = spark.table(f"{catalog_name}.{schema_name}.ravenstack_churn_events").toPandas()
    print(f"loaded Delta {catalog_name}.{schema_name}.ravenstack_*")
except Exception as exc:
    print("Delta indisponivel, lendo CSV:", exc)
    data_path = resolve_data_path(data_path_widget)
    accounts = pd.read_csv(f"{data_path}/ravenstack_accounts.csv")
    subscriptions = pd.read_csv(f"{data_path}/ravenstack_subscriptions.csv")
    feature_usage = pd.read_csv(f"{data_path}/ravenstack_feature_usage.csv")
    support_tickets = pd.read_csv(f"{data_path}/ravenstack_support_tickets.csv")
    churn_events = pd.read_csv(f"{data_path}/ravenstack_churn_events.csv")
    print(f"loaded CSV from {data_path}")

accounts = normalize_frame(accounts)
subscriptions = normalize_frame(subscriptions)
feature_usage = normalize_frame(feature_usage)
support_tickets = normalize_frame(support_tickets)
churn_events = normalize_frame(churn_events)

print({name: frame.shape for name, frame in {
    "accounts": accounts,
    "subscriptions": subscriptions,
    "feature_usage": feature_usage,
    "support_tickets": support_tickets,
    "churn_events": churn_events,
}.items()})


### Create a Table Dictionary

To make the data quality checks easier and more scalable, all loaded tables are stored in a single dictionary.

This allows us to loop through the datasets instead of repeating the same checks manually for each table.

In [ ]:
tables = {
    'accounts': accounts,
    'subscriptions': subscriptions,
    'feature_usage': feature_usage,
    'support_tickets': support_tickets,
    'churn_events': churn_events
}

## Data Quality Audit

Before building churn and retention insights, we first validate whether the dataset is structurally reliable for business analysis.

This audit focuses on identifying potential issues that could impact analytical accuracy, revenue calculations, customer lifecycle tracking, or retention modeling.

We will validate:

- dataset scale and table structure
- missing or incomplete information
- primary and foreign key relationships
- duplicate records
- temporal consistency across events
- business logic anomalies and data integrity issues

In [ ]:
dataset_overview = pd.DataFrame({
    'Table': ['Accounts', 'Subscriptions', 'Feature Usage',
              'Support Tickets', 'Churn Events'],
    
    'Rows': [
        accounts.shape[0],
        subscriptions.shape[0],
        feature_usage.shape[0],
        support_tickets.shape[0],
        churn_events.shape[0]
    ],
    
    'Columns': [
        accounts.shape[1],
        subscriptions.shape[1],
        feature_usage.shape[1],
        support_tickets.shape[1],
        churn_events.shape[1]
    ]
})

dataset_overview

### Initial Observation

The dataset represents a complete SaaS customer lifecycle, including subscriptions, product usage, support activity, and churn events.

Feature Usage is the largest table with 25,000 records, suggesting that customer behavior and product engagement will be key factors in understanding retention and churn risk.

### Missing Value Analysis

Missing values can affect churn analysis, revenue calculations, and customer segmentation.  
Before building business insights, we check which fields contain missing data and whether these gaps are expected or problematic.

In [ ]:
missing_summary = []

for name, table in tables.items():

    missing = table.isnull().sum()
    missing_percent = table.isnull().mean() * 100

    for column in table.columns:

        if missing[column] > 0:

            missing_summary.append({
                'table': name,
                'column': column,
                'missing_values': missing[column],
                'missing_percent': round(
                    missing_percent[column], 2
                )
            })

missing_summary = pd.DataFrame(missing_summary)

missing_summary = missing_summary.sort_values(
    by='missing_percent',
    ascending=False
)

missing_summary

In [ ]:
if not missing_summary.empty:
    plt.figure(figsize=(6, 2.2))

    ax = sns.barplot(
        data=missing_summary,
        x="missing_percent",
        y="column",
        hue="column",
        palette="viridis",
        legend=False
    )

    for container in ax.containers:
        ax.bar_label(
            container,
            fmt="%.1f%%",
            padding=4,
            fontsize=9
        )

    plt.title(
        "Missing Values by Field",
        fontsize=12,
        weight="bold"
    )

    plt.xlabel("Missing values, %")
    plt.ylabel("")

    plt.xticks(fontsize=9)
    plt.yticks(fontsize=10)

    sns.despine(left=True, bottom=True)
    plt.grid(False)
    plt.tight_layout()
    plt.show()

else:
    print("No missing values found across the loaded tables.")

#### Key observations:
- `end_date` is missing for active subscriptions
- `satisfaction_score` is missing when customers do not leave ticket feedback
- `feedback_text` is optional during churn reporting

Overall, missing values appear explainable and do not indicate major structural issues in the dataset.

### Relationship Integrity Validation

Before performing retention and churn analysis, we need to confirm that all operational tables are correctly connected.

The dataset contains multiple related entities: accounts, subscriptions, feature usage, support tickets, and churn events. Broken relationships could distort customer counts, lifecycle tracking, churn attribution, and revenue analysis.

This check validates whether foreign key references remain structurally consistent across the dataset.

In [ ]:
relationship_check = pd.DataFrame({
    "validation_check": [
        "Subscriptions → Accounts",
        "Feature Usage → Subscriptions",
        "Support Tickets → Accounts",
        "Churn Events → Accounts"
    ],
    "invalid_records": [
        len(subscriptions[
            ~subscriptions["account_id"].isin(accounts["account_id"])
        ]),
        len(feature_usage[
            ~feature_usage["subscription_id"].isin(subscriptions["subscription_id"])
        ]),
        len(support_tickets[
            ~support_tickets["account_id"].isin(accounts["account_id"])
        ]),
        len(churn_events[
            ~churn_events["account_id"].isin(accounts["account_id"])
        ])
    ]
})

relationship_check

#### Relationship Validation

All core business tables are correctly connected, with no broken relationships detected across customer, subscription, feature usage, or churn records.

This confirms that the dataset maintains strong relational integrity and can be reliably used for lifecycle, retention, and churn analysis.

### Duplicate Record Validation

Duplicate records can distort SaaS business metrics such as customer counts, churn rate, revenue calculations, and feature adoption.

This check validates whether the dataset contains fully duplicated rows or duplicated primary identifiers across the core business tables.

In [ ]:
duplicate_check = pd.DataFrame({
    "table": list(tables.keys()),
    "duplicate_rows": [
        table.duplicated().sum()
        for table in tables.values()
    ],
    "duplicate_percent": [
        round(table.duplicated().mean() * 100, 2)
        for table in tables.values()
    ]
})

duplicate_check

In [ ]:
duplicate_id_check = pd.DataFrame({
    'id_column': [
        'account_id',
        'subscription_id',
        'usage_id',
        'ticket_id',
        'churn_event_id'
    ],
    'duplicate_ids': [
        accounts['account_id'].duplicated().sum(),
        subscriptions['subscription_id'].duplicated().sum(),
        feature_usage['usage_id'].duplicated().sum(),
        support_tickets['ticket_id'].duplicated().sum(),
        churn_events['churn_event_id'].duplicated().sum()
    ]
})

duplicate_id_check

#### Key Observations

Most primary identifiers are unique across the core business tables.

A small number of duplicated `usage_id` values were found in the `feature_usage` table. Since this affects only a very small share of event-level usage records, it does not materially impact customer-level churn, lifecycle, or revenue analysis.

However, duplicated usage IDs should be handled before running detailed event-level product engagement analysis.

### Business Rule Validation

Business rule validation checks whether the dataset follows realistic SaaS logic.

These checks help identify records that may be technically valid, but analytically questionable, such as churn events without churned accounts, subscriptions with invalid dates, or revenue values that do not match the expected subscription structure.

In [ ]:
# business data issues

business_checks = pd.DataFrame({
    'check': [
        'Negative MRR',
        'Negative ARR',
        'Zero or Negative Seats',
        'Negative Usage Count',
        'Negative Usage Duration',
        'Negative Error Count',
        'Invalid Satisfaction Score',
        'Negative Refund Amount'
    ],
    'invalid_rows': [
        len(subscriptions[subscriptions['mrr_amount'] < 0]),
        len(subscriptions[subscriptions['arr_amount'] < 0]),
        len(subscriptions[subscriptions['seats'] <= 0]),
        len(feature_usage[feature_usage['usage_count'] < 0]),
        len(feature_usage[feature_usage['usage_duration_secs'] < 0]),
        len(feature_usage[feature_usage['error_count'] < 0]),
        len(support_tickets[
            (support_tickets['satisfaction_score'] < 1) |
            (support_tickets['satisfaction_score'] > 5)
        ]),
        len(churn_events[churn_events['refund_amount_usd'] < 0])
    ]
})

business_checks

#### Key Observations

The dataset follows the core SaaS business rules required for this analysis.

No major issues were found that would prevent customer-level churn analysis, subscription lifecycle tracking, or revenue risk evaluation.

This means the dataset is reliable enough to support the next stage of the project: business KPI analysis and retention insight generation.

### Data Quality Conclusion

The dataset is suitable for business-level SaaS retention analysis.

The audit confirmed that:

- core tables are structurally complete
- missing values are explainable and expected
- relationships between tables are valid
- duplicate records are limited and do not affect account-level analysis
- business rules are consistent enough for churn, lifecycle, and revenue analysis

Based on these checks, the dataset can be used to build reliable SaaS KPIs, identify churn drivers, and evaluate revenue risk.

### Analytical Note: Churn Events vs Account Churn

One important distinction in this dataset is the difference between event-level churn data and account-level churn status.

The `churn_events` table contains churn event records, while `accounts.churn_flag` represents the current account-level churn status. Because a customer can have multiple churn events or later reactivation, churn event counts should not be interpreted as the same thing as currently churned customer counts.

For this reason, this analysis uses `accounts.churn_flag` for current churn rate and account-level retention metrics, while `churn_events` is used mainly to understand reported churn reasons and historical churn behavior.

## Executive SaaS KPI Snapshot

After validating the dataset, we can now establish the business baseline.

This snapshot summarizes the current customer base, active accounts, churn rate, trial exposure, and revenue scale. These metrics help frame the rest of the analysis and show where retention efforts may have the highest business impact.

In [ ]:
# KPI calculations

# Make sure date columns are in datetime format
accounts["signup_date"] = pd.to_datetime(accounts["signup_date"])
subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"])

# Account-level KPIs
total_accounts = accounts["account_id"].nunique()

active_accounts = accounts.loc[
    accounts["churn_flag"] == False,
    "account_id"
].nunique()

churn_rate = (
    accounts["churn_flag"].mean() * 100
)

trial_accounts = accounts.loc[
    accounts["is_trial"] == True,
    "account_id"
].nunique()

trial_share = (
    trial_accounts / total_accounts * 100
)

# Current ARR calculation:
# Use only the latest subscription record per account to avoid double-counting historical subscription rows
latest_subscriptions = (
    subscriptions
    .sort_values("start_date")
    .groupby("account_id", as_index=False)
    .tail(1)
)

active_subscription_arr = latest_subscriptions.loc[
    latest_subscriptions["churn_flag"] == False,
    "arr_amount"
].sum()

In [ ]:
from matplotlib.patches import FancyBboxPatch

# Executive KPI cards

kpi_cards = [
    ("Total Accounts", f"{total_accounts:,.0f}", "customer base"),
    ("Active Accounts", f"{active_accounts:,.0f}", "currently active"),
    ("Churn Rate", f"{churn_rate:.1f}%", "account-level churn"),
    ("Trial Accounts", f"{trial_accounts:,.0f}", "trial exposure"),
    ("Trial Share", f"{trial_share:.1f}%", "of total accounts"),
    (
        "Current ARR",
        f"${active_subscription_arr / 1_000_000:.1f}M",
        "latest active subscriptions"
    )
]

colors = sns.color_palette("viridis", len(kpi_cards)).as_hex()

fig = plt.figure(figsize=(12, 3.4))
fig.patch.set_facecolor(plt.rcParams["figure.facecolor"])

fig.text(
    0.5, 0.92,
    "Executive SaaS KPI Snapshot",
    ha="center",
    va="center",
    fontsize=16,
    fontweight="bold",
    color=plt.rcParams["text.color"]
)

card_width = 0.27
card_height = 0.27

x_positions = [0.06, 0.365, 0.67]
y_positions = [0.55, 0.18]

for i, (title, value, subtitle) in enumerate(kpi_cards):
    row = i // 3
    col = i % 3

    ax = fig.add_axes([
        x_positions[col],
        y_positions[row],
        card_width,
        card_height
    ])

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    card = FancyBboxPatch(
        (0.02, 0.05),
        0.96,
        0.90,
        boxstyle="round,pad=0.025,rounding_size=0.06",
        linewidth=1,
        edgecolor=colors[i],
        facecolor=plt.rcParams["axes.facecolor"],
        alpha=0.95
    )

    ax.add_patch(card)

    ax.text(
        0.5, 0.63,
        value,
        ha="center",
        va="center",
        fontsize=24,
        fontweight="bold",
        color=colors[i]
    )

    ax.text(
        0.5, 0.37,
        title,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color=plt.rcParams["text.color"]
    )

    ax.text(
        0.5, 0.20,
        subtitle,
        ha="center",
        va="center",
        fontsize=8.5,
        color=plt.rcParams["axes.labelcolor"],
        alpha=0.75
    )

plt.show()

#### Executive Insight

RavenStack has 500 accounts, with 390 currently active and an overall account churn rate of 22.0%.

This indicates a meaningful retention challenge. However, churn should not be treated as one single number. The next step is to understand where churn is concentrated and whether it is connected to trial status, subscription changes, support experience, product usage, or revenue exposure.

## Customer Lifecycle Risk Map

After the executive KPI snapshot, the next step is to understand where retention risk appears across the customer lifecycle.

Instead of only showing how customers move through the journey, this view compares lifecycle segments by churn rate and segment size. This helps identify which customer paths deserve more attention from the business.

Each segment combines three signals:

- entry type: trial or direct paid
- initial plan tier: Basic, Pro, or Enterprise
- subscription movement: upgraded, downgraded, or stable

In [ ]:
# Customer lifecycle risk map

account_movement = (
    subscriptions
    .groupby("account_id")
    .agg(
        had_upgrade=("upgrade_flag", "max"),
        had_downgrade=("downgrade_flag", "max")
    )
    .reset_index()
)

lifecycle_df = accounts.merge(
    account_movement,
    on="account_id",
    how="left"
)

lifecycle_df["entry_type"] = lifecycle_df["is_trial"].map({
    True: "Trial",
    False: "Direct Paid"
})

lifecycle_df["movement_type"] = "Stable"

lifecycle_df.loc[
    lifecycle_df["had_upgrade"] == True,
    "movement_type"
] = "Upgraded"

lifecycle_df.loc[
    lifecycle_df["had_downgrade"] == True,
    "movement_type"
] = "Downgraded"

lifecycle_risk = (
    lifecycle_df
    .groupby(["entry_type", "plan_tier", "movement_type"], observed=True)
    .agg(
        accounts=("account_id", "nunique"),
        churned_accounts=("churn_flag", "sum")
    )
    .reset_index()
)

lifecycle_risk["churn_rate"] = (
    lifecycle_risk["churned_accounts"]
    / lifecycle_risk["accounts"]
    * 100
)

lifecycle_risk["lifecycle_path"] = (
    lifecycle_risk["entry_type"]
    + " → "
    + lifecycle_risk["plan_tier"]
    + " → "
    + lifecycle_risk["movement_type"]
)

lifecycle_risk = lifecycle_risk.sort_values(
    by="churn_rate",
    ascending=False
)

In [ ]:
# Lifecycle risk priority dot plot

min_accounts = 10

top_lifecycle_risk = (
    lifecycle_risk
    .loc[lifecycle_risk["accounts"] >= min_accounts]
    .copy()
)

top_lifecycle_risk["risk_score"] = (
    top_lifecycle_risk["churn_rate"]
    * np.log1p(top_lifecycle_risk["accounts"])
)

top_lifecycle_risk = (
    top_lifecycle_risk
    .sort_values("risk_score", ascending=False)
    .head(10)
    .sort_values("churn_rate", ascending=True)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(12, 5.6))
fig.patch.set_facecolor(plt.rcParams["figure.facecolor"])
ax.set_facecolor(plt.rcParams["axes.facecolor"])

colors = sns.color_palette(
    "viridis",
    len(top_lifecycle_risk)
).as_hex()

y_positions = range(len(top_lifecycle_risk))

ax.scatter(
    top_lifecycle_risk["churn_rate"],
    y_positions,
    c=colors,
    s=180,
    alpha=0.95,
    edgecolors=plt.rcParams["text.color"],
    linewidths=0.6
)

x_max = top_lifecycle_risk["churn_rate"].max()
segment_size_x = x_max + 3.0

for y, row in zip(y_positions, top_lifecycle_risk.itertuples()):
    ax.text(
        row.churn_rate + 0.35,
        y,
        f"{row.churn_rate:.1f}%",
        va="center",
        fontsize=9.5,
        color=plt.rcParams["text.color"]
    )

    ax.text(
        segment_size_x,
        y,
        f"{row.accounts} accounts",
        va="center",
        fontsize=9,
        color=plt.rcParams["axes.labelcolor"],
        alpha=0.85
    )

ax.text(
    segment_size_x,
    len(top_lifecycle_risk) - 0.35,
    "Segment size",
    ha="left",
    va="bottom",
    fontsize=9,
    fontweight="bold",
    color=plt.rcParams["text.color"],
    alpha=0.9
)

ax.set_yticks(list(y_positions))
ax.set_yticklabels(
    top_lifecycle_risk["lifecycle_path"],
    fontsize=10
)

ax.set_title(
    "Lifecycle Segments with the Highest Retention Risk",
    fontsize=16,
    fontweight="bold",
    pad=16
)

ax.set_xlabel("Churn rate (%)")
ax.set_ylabel("")

ax.set_xlim(
    top_lifecycle_risk["churn_rate"].min() - 1,
    x_max + 5
)

ax.grid(False)
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.show()

#### Business Insight

The lifecycle risk map shows that the highest churn rate is not always the highest business priority.

Some smaller trial-based segments show elevated churn, but their account volume is limited. Larger direct-paid segments with meaningful churn rates may represent more important retention opportunities because they affect more customers.

For example, `Direct Paid → Enterprise → Upgraded` combines a high churn rate with a relatively large segment size, making it more important to investigate than smaller trial-only paths.

This suggests that RavenStack should prioritize retention efforts based on both churn rate and segment size, not churn rate alone. The next step is to investigate what is driving churn inside these high-priority lifecycle segments.

## Churn Reasons Analysis

After identifying lifecycle segments with higher retention risk, the next step is to understand why customers leave.

This section uses the `churn_events` table to analyze reported churn reasons. Since this table is event-level, the analysis focuses on unique accounts per reason where possible, instead of treating every churn event as a separate customer.

In [ ]:
# Analyze reported churn reasons
# Count unique accounts per reason to avoid treating repeated churn events as separate customers

reason_summary = (
    churn_events
    .groupby("reason_code")
    .agg(
        unique_accounts=("account_id", "nunique"),
        churn_events=("churn_event_id", "nunique")
    )
    .reset_index()
    .sort_values("unique_accounts", ascending=False)
)

reason_summary["reason_label"] = (
    reason_summary["reason_code"]
    .str.replace("_", " ")
    .str.title()
)

plot_data = reason_summary.sort_values(
    "unique_accounts",
    ascending=True
)

fig, ax = plt.subplots(figsize=(10, 4.8))
fig.patch.set_facecolor(plt.rcParams["figure.facecolor"])
ax.set_facecolor(plt.rcParams["axes.facecolor"])

colors = sns.color_palette(
    "viridis",
    len(plot_data)
).as_hex()

ax.barh(
    plot_data["reason_label"],
    plot_data["unique_accounts"],
    color=colors,
    edgecolor=plt.rcParams["text.color"],
    linewidth=0.5,
    alpha=0.9
)

for index, value in enumerate(plot_data["unique_accounts"]):
    ax.text(
        value + 1,
        index,
        f"{value}",
        va="center",
        fontsize=10,
        color=plt.rcParams["text.color"]
    )

ax.set_title(
    "Top Reported Churn Reasons by Unique Accounts",
    fontsize=16,
    fontweight="bold",
    pad=16
)

ax.set_xlabel("Number of unique accounts")
ax.set_ylabel("")

ax.set_xlim(0, plot_data["unique_accounts"].max() + 8)

ax.grid(False)
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.show()

#### Business Insight

Feature limitations are the most frequently reported churn reason, followed closely by budget and support issues.

However, the gap between the top reasons is not large. This suggests that churn is not driven by one isolated problem, but by a combination of product value, support experience, and pricing or budget pressure.

For RavenStack, the first retention priority should be to investigate feature-related complaints, but support and budget-related churn should be analyzed alongside them because they represent almost the same level of churn activity.

## Subscription Change Signals

Plan changes can reveal early warning signs of customer risk.

This analysis looks at subscription movement at the account level. Instead of counting every subscription record, each account is classified by whether it only upgraded, only downgraded, stayed stable, or had both upgrade and downgrade activity.

This helps us understand whether subscription changes can be used as a practical retention signal before customers fully churn.

In [ ]:
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import Normalize

# Subscription change signals

subscription_movement = (
    subscriptions
    .groupby("account_id")
    .agg(
        had_upgrade=("upgrade_flag", "max"),
        had_downgrade=("downgrade_flag", "max")
    )
    .reset_index()
)

subscription_movement = accounts.merge(
    subscription_movement,
    on="account_id",
    how="left"
)

def classify_subscription_change(row):
    if row["had_upgrade"] and row["had_downgrade"]:
        return "Both upgrade\nand downgrade"
    if row["had_upgrade"]:
        return "Upgraded only"
    if row["had_downgrade"]:
        return "Downgraded only"
    return "Stable"

subscription_movement["subscription_change"] = subscription_movement.apply(
    classify_subscription_change,
    axis=1
)

subscription_change_summary = (
    subscription_movement
    .groupby("subscription_change")
    .agg(
        accounts=("account_id", "nunique"),
        churned_accounts=("churn_flag", "sum")
    )
    .reset_index()
)

subscription_change_summary["churn_rate"] = (
    subscription_change_summary["churned_accounts"]
    / subscription_change_summary["accounts"]
    * 100
).round(1)

subscription_change_summary = subscription_change_summary.sort_values(
    "churn_rate",
    ascending=False
).reset_index(drop=True)

# Visual: subscription change risk tiles

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
fig.patch.set_facecolor(plt.rcParams["figure.facecolor"])

norm = Normalize(
    vmin=subscription_change_summary["churn_rate"].min(),
    vmax=subscription_change_summary["churn_rate"].max()
)

cmap = plt.cm.viridis

for ax, row in zip(axes, subscription_change_summary.itertuples()):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    card_color = cmap(norm(row.churn_rate))

    card = FancyBboxPatch(
        (0.04, 0.08),
        0.92,
        0.84,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.1,
        edgecolor=plt.rcParams["text.color"],
        facecolor=card_color,
        alpha=0.92
    )

    ax.add_patch(card)

    ax.text(
        0.5, 0.70,
        f"{row.churn_rate:.1f}%",
        ha="center",
        va="center",
        fontsize=25,
        fontweight="bold",
        color="white"
    )

    ax.text(
        0.5, 0.49,
        row.subscription_change,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color="white"
    )

    ax.text(
        0.5, 0.30,
        f"{row.churned_accounts} churned / {row.accounts} accounts",
        ha="center",
        va="center",
        fontsize=8.8,
        color="white",
        alpha=0.9
    )

    ax.text(
        0.5, 0.18,
        "account-level signal",
        ha="center",
        va="center",
        fontsize=8,
        color="white",
        alpha=0.75
    )

fig.text(
    0.5, 0.98,
    "Subscription Change Signals by Churn Rate",
    ha="center",
    va="center",
    fontsize=16,
    fontweight="bold",
    color=plt.rcParams["text.color"]
)

plt.subplots_adjust(
    top=0.82,
    bottom=0.05,
    left=0.03,
    right=0.97,
    wspace=0.15
)

plt.show()

#### Business Insight

Subscription movement is a useful retention signal, but it should not be interpreted too simply.

Accounts that only downgraded show the highest churn rate at 30.0%. This suggests that downgrade behavior may reflect lower product value, budget pressure, or reduced confidence in the platform.

However, upgraded accounts also show a meaningful churn rate at 23.7%. This means an upgrade is not automatically a retention win. Some customers may upgrade because they expect more value, but still leave if the product does not meet those expectations.

For RavenStack, downgrade activity should be monitored as a high-risk signal. At the same time, upgraded accounts should be checked after the plan change for product adoption, support issues, and satisfaction.

# Trial vs Paid Customer Churn

Trial customers often behave differently from paid customers.

This analysis checks whether trial accounts show higher churn risk than paying customers.

In [ ]:
# Trial vs paid churn benchmark

trial_retention = (
    accounts
    .assign(
        entry_type=accounts["is_trial"].map({
            True: "Trial Accounts",
            False: "Direct Paid Accounts"
        })
    )
    .groupby("entry_type")
    .agg(
        total_accounts=("account_id", "nunique"),
        churned_accounts=("churn_flag", "sum")
    )
    .reset_index()
)

trial_retention["churn_rate"] = (
    trial_retention["churned_accounts"]
    / trial_retention["total_accounts"]
    * 100
).round(1)

trial_retention = (
    trial_retention
    .sort_values("churn_rate", ascending=True)
    .reset_index(drop=True)
)

overall_churn_rate = round(accounts["churn_flag"].mean() * 100, 1)

fig, ax = plt.subplots(figsize=(10.5, 3.4))
fig.patch.set_facecolor(plt.rcParams["figure.facecolor"])
ax.set_facecolor(plt.rcParams["axes.facecolor"])

colors = sns.color_palette("viridis", len(trial_retention)).as_hex()
y_positions = range(len(trial_retention))

# Overall benchmark line
ax.axvline(
    overall_churn_rate,
    color=plt.rcParams["axes.labelcolor"],
    linestyle="--",
    linewidth=1.4,
    alpha=0.55
)

# Main lines and points
for y, row, color in zip(y_positions, trial_retention.itertuples(), colors):
    ax.hlines(
        y=y,
        xmin=0,
        xmax=row.churn_rate,
        color=color,
        linewidth=7,
        alpha=0.85
    )

    ax.scatter(
        row.churn_rate,
        y,
        s=230,
        color=color,
        edgecolors=plt.rcParams["text.color"],
        linewidths=0.7,
        zorder=3
    )

    ax.text(
        row.churn_rate + 0.6,
        y,
        f"{row.churn_rate:.1f}%",
        va="center",
        fontsize=11,
        fontweight="bold",
        color=plt.rcParams["text.color"]
    )

# Right-side account counts
count_x = 32.5

for y, row in zip(y_positions, trial_retention.itertuples()):
    ax.text(
        count_x,
        y,
        f"{row.churned_accounts} churned / {row.total_accounts} accounts",
        va="center",
        fontsize=9,
        color=plt.rcParams["axes.labelcolor"],
        alpha=0.85
    )

ax.text(
    count_x,
    max(y_positions) + 0.28,
    "Segment size",
    ha="left",
    va="bottom",
    fontsize=9,
    fontweight="bold",
    color=plt.rcParams["text.color"],
    alpha=0.9
)

# Benchmark label
ax.text(
    overall_churn_rate + 0.35,
    -0.38,
    f"Overall churn benchmark: {overall_churn_rate:.1f}%",
    fontsize=9,
    color=plt.rcParams["axes.labelcolor"],
    alpha=0.85
)

ax.set_yticks(list(y_positions))
ax.set_yticklabels(
    trial_retention["entry_type"],
    fontsize=10
)

ax.set_title(
    "Trial vs Direct Paid Churn Risk",
    fontsize=16,
    fontweight="bold",
    pad=14
)

ax.set_xlabel("Churn rate (%)")
ax.set_ylabel("")

ax.set_xlim(0, 38)
ax.set_ylim(-0.55, 1.45)

ax.grid(False)
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.show()

#### Business Insight

Trial accounts churn at a higher rate than direct paid accounts.

Trial customers show a churn rate of 25.8%, compared with 21.1% for direct paid accounts and 22.0% overall. This means trial onboarding creates additional retention risk.

However, direct paid accounts still represent a much larger customer base: 403 accounts compared with 97 trial accounts. RavenStack should improve trial activation, but retention work should not focus only on trial users. Paid customers also need monitoring around product value, support experience, and post-purchase adoption.

## Geographic Retention Priority

Not every high-churn market has the same business impact.

A country with a high churn rate may look risky, but it may not be the biggest business priority if the customer base is small.

This view ranks countries by churned customer volume and adds churn rate as context. The goal is to identify where geographic retention work can protect the largest number of customers.

In [ ]:
# Geographic retention priority

country_risk = (
    accounts
    .groupby("country")
    .agg(
        customers=("account_id", "nunique"),
        churned_customers=("churn_flag", "sum"),
        churn_rate=("churn_flag", "mean")
    )
    .reset_index()
)

country_risk["churn_rate"] = (country_risk["churn_rate"] * 100).round(1)

country_risk = (
    country_risk
    .sort_values("churned_customers", ascending=True)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(9, 4.8))
fig.patch.set_facecolor(plt.rcParams["figure.facecolor"])
ax.set_facecolor(plt.rcParams["axes.facecolor"])

colors = sns.color_palette("viridis", len(country_risk)).as_hex()

ax.barh(
    country_risk["country"],
    country_risk["churned_customers"],
    color=colors,
    alpha=0.9
)

for row in country_risk.itertuples():
    ax.text(
        row.churned_customers + 1,
        row.country,
        f"{row.churned_customers} churned · {row.churn_rate:.1f}% churn",
        va="center",
        fontsize=9.5,
        color=plt.rcParams["text.color"]
    )

ax.set_title(
    "Geographic Retention Priority by Market",
    fontsize=16,
    fontweight="bold",
    pad=14
)

ax.set_xlabel("Churned customers")
ax.set_ylabel("")

ax.set_xlim(0, country_risk["churned_customers"].max() + 18)

ax.grid(False)
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.show()

#### Business Insight

The geographic analysis shows that retention priority should be based on customer impact, not churn rate alone.

Germany has the highest churn rate at 32.0%, but the United States creates the largest business impact with 68 churned customers and a 23.4% churn rate.

This means RavenStack should investigate the US market first, while smaller high-churn countries such as Germany should be treated as targeted follow-up segments in the broader retention analysis.

## Cohort Retention Analysis

To understand how retention changes across customer lifecycle stages, I grouped accounts by signup month and tracked whether they remained active in each following month.

Because the churn table is event-level and some accounts can have multiple churn events, this cohort analysis uses the first churn date per account. This prevents repeated churn events from duplicating the same customer inside the retention matrix.

In [ ]:
# Prepare cohort data

cohort_df = accounts.copy()

cohort_df["signup_date"] = pd.to_datetime(cohort_df["signup_date"])
cohort_df["cohort_month"] = cohort_df["signup_date"].dt.to_period("M")


# Use the first churn date per account
# This prevents accounts with multiple churn events from being duplicated in the cohort analysis

first_churn_dates = (
    churn_events
    .assign(churn_date=pd.to_datetime(churn_events["churn_date"]))
    .groupby("account_id", as_index=False)
    .agg(first_churn_date=("churn_date", "min"))
)


cohort_df = cohort_df.merge(
    first_churn_dates,
    on="account_id",
    how="left"
)


# Define observation end date

observation_end = max(
    cohort_df["signup_date"].max(),
    cohort_df["first_churn_date"].max()
)


# Build monthly retention records

retention_records = []

for _, row in cohort_df.iterrows():
    signup_month = row["cohort_month"]
    churn_date = row["first_churn_date"]

    max_months = (
        observation_end.to_period("M") - signup_month
    ).n

    for month_number in range(max_months + 1):
        current_month = signup_month + month_number
        current_month_end = current_month.to_timestamp(how="end")

        is_active = (
            pd.isna(churn_date)
            or churn_date > current_month_end
        )

        retention_records.append({
            "cohort_month": str(signup_month),
            "month_number": month_number,
            "account_id": row["account_id"],
            "is_active": is_active
        })


retention_df = pd.DataFrame(retention_records)


# Calculate monthly retention rate by signup cohort

cohort_retention = (
    retention_df
    .groupby(["cohort_month", "month_number"])["is_active"]
    .mean()
    .reset_index()
)


cohort_matrix = cohort_retention.pivot(
    index="cohort_month",
    columns="month_number",
    values="is_active"
)


# Keep chart compact: latest 8 cohorts and first 8 lifecycle months

cohort_matrix = cohort_matrix.iloc[-8:, :8]


# Plot heatmap

fig, ax = plt.subplots(figsize=(9, 4.8))

fig.patch.set_facecolor(plt.rcParams["figure.facecolor"])
ax.set_facecolor(plt.rcParams["axes.facecolor"])

sns.heatmap(
    cohort_matrix,
    annot=True,
    fmt=".0%",
    cmap="viridis",
    linewidths=0.5,
    linecolor=plt.rcParams["axes.facecolor"],
    cbar_kws={"label": "Retention Rate"},
    ax=ax
)

ax.set_title(
    "Monthly Cohort Retention Heatmap",
    fontsize=16,
    fontweight="bold",
    color=plt.rcParams["text.color"],
    pad=14
)

ax.set_xlabel(
    "Months Since Signup",
    fontsize=10,
    color=plt.rcParams["axes.labelcolor"]
)

ax.set_ylabel(
    "Signup Cohort",
    fontsize=10,
    color=plt.rcParams["axes.labelcolor"]
)

ax.tick_params(
    colors=plt.rcParams["axes.labelcolor"],
    labelsize=9
)

# Style colorbar text for dark theme
colorbar = ax.collections[0].colorbar
colorbar.ax.yaxis.label.set_color(plt.rcParams["axes.labelcolor"])
colorbar.ax.tick_params(colors=plt.rcParams["axes.labelcolor"])

plt.tight_layout()
plt.show()

### Insight

The cohort heatmap helps identify where retention starts to weaken after signup.

Instead of looking only at total churn, this view shows how customer retention changes over time for different signup cohorts. This makes it easier to spot lifecycle stages where onboarding, activation, or customer success actions may have the strongest impact.

## Support Experience and Churn Risk

After analyzing how retention changes over time, the next question is whether customer support experience is connected to churn.

For a SaaS business, slow resolution times and low satisfaction scores can become early warning signals of customer frustration.

This section groups accounts by support resolution speed and satisfaction level to check whether weaker support experience is associated with higher churn risk.

In [ ]:
# Support risk heatmap

support_account_summary = (
    support_tickets
    .groupby("account_id")
    .agg(
        tickets=("ticket_id", "nunique"),
        avg_resolution_hours=("resolution_time_hours", "mean"),
        avg_satisfaction=("satisfaction_score", "mean")
    )
    .reset_index()
)

support_churn = accounts.merge(
    support_account_summary,
    on="account_id",
    how="inner"
)

support_churn = support_churn.dropna(
    subset=["avg_resolution_hours", "avg_satisfaction"]
).copy()

support_churn["resolution_group"] = pd.qcut(
    support_churn["avg_resolution_hours"],
    q=3,
    labels=["Fast", "Medium", "Slow"]
)

support_churn["satisfaction_group"] = pd.cut(
    support_churn["avg_satisfaction"],
    bins=[0, 3.5, 4.5, 5],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

support_risk = (
    support_churn
    .groupby(
        ["satisfaction_group", "resolution_group"],
        observed=True
    )
    .agg(
        accounts=("account_id", "nunique"),
        churn_rate=("churn_flag", "mean")
    )
    .reset_index()
)

support_risk["churn_rate"] = (
    support_risk["churn_rate"] * 100
).round(1)

heatmap_values = support_risk.pivot(
    index="satisfaction_group",
    columns="resolution_group",
    values="churn_rate"
)

heatmap_counts = support_risk.pivot(
    index="satisfaction_group",
    columns="resolution_group",
    values="accounts"
)

heatmap_values = heatmap_values.reindex(["High", "Medium", "Low"])
heatmap_counts = heatmap_counts.reindex(["High", "Medium", "Low"])

annotations = heatmap_values.copy()

for row in annotations.index:
    for col in annotations.columns:
        churn_value = heatmap_values.loc[row, col]
        account_count = heatmap_counts.loc[row, col]

        annotations.loc[row, col] = (
            f"{churn_value:.1f}%\n"
            f"{int(account_count)} accounts"
        )

fig, ax = plt.subplots(figsize=(8.2, 4.4))
fig.patch.set_facecolor(plt.rcParams["figure.facecolor"])
ax.set_facecolor(plt.rcParams["axes.facecolor"])

sns.heatmap(
    heatmap_values,
    annot=annotations,
    fmt="",
    cmap="viridis",
    linewidths=1.2,
    linecolor=plt.rcParams["figure.facecolor"],
    cbar=False,
    ax=ax
)

ax.set_title(
    "Churn Rate by Support Experience Segment",
    fontsize=16,
    fontweight="bold",
    pad=14
)

ax.set_xlabel("Resolution time group")
ax.set_ylabel("Satisfaction level")

ax.tick_params(
    axis="both",
    labelsize=10,
    colors=plt.rcParams["axes.labelcolor"]
)

plt.tight_layout()
plt.show()

#### Business Insight

The support heatmap does not show a simple pattern where slow resolution and low satisfaction automatically lead to the highest churn.

The highest churn rates appear in medium-resolution segments, while slow-resolution segments do not show the highest churn. This suggests that support experience alone does not fully explain customer churn.

For RavenStack, support should still be monitored as a retention signal, but it should be analyzed together with churn reasons, feature limitations, pricing pressure, lifecycle stage, and revenue exposure.

This is especially important because support appeared as one of the top reported churn reasons, even though broad support metrics do not show a simple direct pattern.

## Revenue Exposure by Industry

Not all customer segments contribute equally to business risk.

After analyzing churn through lifecycle, geography, cohort behavior, and support experience, the next step is to connect retention risk with revenue exposure.

This treemap compares industries by active subscription ARR and churn rate to identify which segments combine high revenue contribution with elevated retention risk.

In [ ]:
# Revenue exposure by industry
# Use latest subscription per account to avoid double-counting historical subscription rows

subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"])

latest_subscriptions = (
    subscriptions
    .sort_values("start_date")
    .groupby("account_id", as_index=False)
    .tail(1)
)

active_subscriptions = latest_subscriptions[
    latest_subscriptions["churn_flag"] == False
].copy()

industry_risk = (
    active_subscriptions
    .merge(
        accounts[["account_id", "industry", "churn_flag"]],
        on="account_id",
        how="left",
        suffixes=("_subscription", "_account")
    )
    .groupby("industry")
    .agg(
        active_arr=("arr_amount", "sum"),
        customers=("account_id", "nunique"),
        churn_rate=("churn_flag_account", "mean")
    )
    .reset_index()
)

industry_risk["churn_rate"] = (
    industry_risk["churn_rate"] * 100
).round(1)

industry_risk["active_arr_m"] = (
    industry_risk["active_arr"] / 1_000_000
).round(1)

fig = px.treemap(
    industry_risk,
    path=["industry"],
    values="active_arr",
    color="churn_rate",
    color_continuous_scale="Viridis",
    custom_data=[
        "active_arr_m",
        "customers",
        "churn_rate"
    ],
    title="Revenue Exposure and Churn Risk by Industry"
)

fig.update_traces(
    texttemplate=(
        "<b>%{label}</b><br>"
        "$%{customdata[0]:.1f}M ARR<br>"
        "%{customdata[2]:.1f}% churn"
    ),
    textfont_size=14,
    marker=dict(
        line=dict(
            width=1,
            color="#111111"
        )
    ),
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Current ARR: $%{customdata[0]:.1f}M<br>"
        "Customers: %{customdata[1]}<br>"
        "Churn rate: %{customdata[2]:.1f}%"
        "<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_dark",
    title={
        "text": "Revenue Exposure and Churn Risk by Industry",
        "x": 0.5,
        "font": dict(size=20)
    },
    margin=dict(
        t=60,
        l=10,
        r=10,
        b=10
    ),
    height=520,
    coloraxis_colorbar=dict(
        title="Churn rate (%)"
    )
)

fig.show()

#### Business Insight

FinTech represents the largest current ARR exposure among active accounts, while DevTools shows the highest churn rate among the major industry segments.

This means RavenStack should not prioritize industries by revenue size alone. A segment with slightly lower revenue but higher churn risk can still require earlier retention attention.

DevTools should be investigated as a high-risk segment because it combines meaningful revenue exposure with elevated churn. FinTech should also remain a priority because of its larger current revenue base.

## Business Recommendations

### 1. Monitor downgrade activity as an early churn signal

Accounts that only downgraded showed the highest churn rate among subscription movement groups.

This suggests that downgrade behavior may indicate reduced product value, budget pressure, or lower confidence in the platform.

RavenStack should flag downgraded accounts for proactive customer success follow-up before they fully churn.

### 2. Strengthen early lifecycle onboarding

The cohort retention heatmap shows that retention weakens as customers move further from signup, with some newer cohorts showing faster early drop-off.

RavenStack should review onboarding, activation, and early feature education, especially for recent signup cohorts.

The goal is to help new customers reach product value faster during the first stages of the customer lifecycle.

### 3. Treat support as a contextual retention signal

Support appeared as one of the top reported churn reasons, but the support heatmap did not show a simple pattern where slow resolution and low satisfaction automatically produced the highest churn.

This means support should not be used as a standalone churn predictor.

Instead, RavenStack should combine support signals with churn reasons, feature complaints, pricing pressure, lifecycle stage, and revenue exposure.

### 4. Prioritize retention by business impact, not churn rate alone

The geographic analysis showed that the highest churn rate is not always the highest business priority.

Germany had the highest churn rate, but the United States created the largest customer impact because it had the highest number of churned customers.

Retention campaigns should prioritize markets and segments where churn rate, customer volume, and revenue exposure are all meaningful.

### 5. Focus first on high-revenue segments with elevated churn risk

The industry revenue analysis showed that DevTools is a high-priority segment because it combines large active ARR with the highest churn rate.

RavenStack should investigate DevTools accounts more deeply by connecting churn reasons, product usage, support experience, and subscription changes inside this segment.

This can help the business protect revenue where churn creates the largest financial risk.

### 6. Move from reactive churn reporting to proactive retention management

Instead of only reporting churn after it happens, RavenStack should build a simple customer health monitoring system.

This system should combine:

- subscription movement
- trial status
- cohort behavior
- geography
- support experience
- churn reasons
- revenue exposure

The goal is to identify at-risk customers earlier and focus retention actions where they can protect the most revenue.

## Conclusion

This analysis shows that RavenStack’s churn problem should not be treated as one single final outcome.

Retention risk appears earlier through subscription movement, trial behavior, cohort performance, geography, support signals, and revenue exposure.

The most important takeaway is that churn priority should be based not only on churn rate, but also on business impact. Some segments have high churn but limited customer volume, while others combine meaningful revenue exposure with elevated retention risk.

For RavenStack, the strongest next step is to move from reactive churn reporting to proactive retention management by combining lifecycle signals, customer experience data, and revenue exposure into a simple customer health monitoring system.